# 02 — Chatbot with the Gemini API

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mano-octavianojr/chatbot-for-insurance-knowledge/blob/main/notebooks/02_gemini_api_chatbot.ipynb)

Run cells from top to bottom. This lesson uses fictional insurance examples for education, not coverage advice. Do not enter personal, medical, or real policy information. No GPU is needed.

## Goal
Use an AI API, system instructions, and conversation history. Unlike lesson 1, this bot generates answers rather than selecting fixed text. It can also make mistakes.

## Connect to Gemini

Create a key in [Google AI Studio](https://aistudio.google.com/apikey). In Colab, open **Secrets** (key icon), add `GEMINI_API_KEY`, and enable notebook access. Never paste the key into a code cell. Outside Colab, use an environment variable with that name.

Gemini offers a free tier for eligible models, accounts, and regions; quotas, availability, and pricing can change. Check [pricing](https://ai.google.dev/gemini-api/docs/pricing) and [rate limits](https://ai.google.dev/gemini-api/docs/rate-limits) before running. Your prompts and documents are sent to Google; review its data-use terms. These cells need Internet access.

In [ ]:
%pip -q install "google-genai>=2.0,<3.0"

In [ ]:
import os
from google import genai
from google.genai import types

def read_api_key():
    key = os.environ.get("GEMINI_API_KEY")
    if key:
        return key
    try:
        from google.colab import userdata
        key = userdata.get("GEMINI_API_KEY")
    except ImportError:
        raise RuntimeError("Set GEMINI_API_KEY in your environment.") from None
    except Exception:
        raise RuntimeError("Add GEMINI_API_KEY to Colab Secrets and enable notebook access.") from None
    if not key:
        raise RuntimeError("GEMINI_API_KEY is missing.")
    return key

client = genai.Client(api_key=read_api_key())
# Change this if your account requires a different supported model.
MODEL = "gemini-flash-latest"


If a request fails: check key permissions for authentication errors, choose an available model for model errors, and wait or reduce requests for quota errors (429). Do not repeatedly retry unchanged requests. Successful responses do not guarantee factual accuracy.

In [ ]:
SYSTEM = """You are an insurance education assistant. Explain general concepts clearly and briefly.
Use fictional examples. Do not determine actual coverage or promise claim approval.
Ask for clarification when needed and acknowledge uncertainty.
Never request sensitive personal information. Actual coverage depends on policy terms."""

response = client.models.generate_content(
    model=MODEL,
    contents="Explain a premium and a deductible with a fictional example.",
    config=types.GenerateContentConfig(system_instruction=SYSTEM),
)
print(response.text or "No text returned; inspect response feedback.")
if response.usage_metadata:
    print("Token usage:", response.usage_metadata)


## Maintain conversation history
The chat object sends conversation context on later turns. Longer histories consume more tokens. Resetting creates a new conversation.

In [ ]:
def new_chat():
    return client.chats.create(
        model=MODEL,
        config=types.GenerateContentConfig(system_instruction=SYSTEM),
    )

chat_session = new_chat()
print(chat_session.send_message("What is a deductible?").text)
print(chat_session.send_message("Can you give a simple example of that?").text)


In [ ]:
def chat():
    session = new_chat()
    print("Type exit to stop or reset to clear conversation history.")
    while True:
        message = input("You: ").strip()
        if message.lower() == "exit":
            break
        if message.lower() == "reset":
            session = new_chat()
            print("Conversation reset.")
            continue
        if message:
            print("Bot:", session.send_message(message).text)

# Uncomment when ready. Each turn makes an API request:
# chat()


## Evaluate
Try “Will my insurer approve my claim?” and an ambiguous question such as “Is it covered?” Check whether the bot avoids guarantees and asks for clarification. These are manual checks, not proof of reliability.

**Exercise:** ask for beginner-friendly explanations through the system instruction and compare responses.

**Next:** ground answers in supplied documents rather than relying only on model knowledge.